# 01 — Data Loading & Cleaning

This notebook ingests all nine raw Olist CSV files, examines their structure,
joins them into a single order-level master table, applies cleaning rules,
engineers cohort and time features, and saves the result as Parquet for use
in every downstream notebook.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path('..').resolve()))

from src.data_processing import (
    load_olist_data, build_master_table, cast_dtypes,
    clean_master_table, add_time_features, add_order_level_features,
    validate_data, save_processed,
)

DATA_DIR  = Path('../data/raw')
PROC_PATH = Path('../data/processed/orders_master.parquet')
FIG_DIR   = Path('../outputs/figures')
TBL_DIR   = Path('../outputs/tables')

for d in (PROC_PATH.parent, FIG_DIR, TBL_DIR):
    d.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.4f}'.format)
print('Setup complete.')

## 1. Load Data

We read all nine Olist CSV files into a dictionary of DataFrames keyed by table name.
Nothing is joined yet — this step is purely about getting the raw data into memory
and confirming every file loaded without errors.

In [ ]:
dfs = load_olist_data(DATA_DIR)

shape_summary = pd.DataFrame(
    [(name, df.shape[0], df.shape[1]) for name, df in dfs.items()],
    columns=['table', 'rows', 'cols'],
).sort_values('rows', ascending=False).reset_index(drop=True)

print(f'Loaded {len(dfs)} tables\n')
print(shape_summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
s = shape_summary.sort_values('rows')
ax.barh(s['table'], s['rows'], color='#3498db', edgecolor='none', height=0.6)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.set_xlabel('Row Count', fontsize=11)
ax.set_title('Row Counts per Olist Table', fontsize=13, fontweight='bold')
ax.grid(True, axis='x', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / '01_table_row_counts.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Explore Schemas

Before joining anything we inspect column names, data types, and null rates for each
table. This tells us which columns need type-casting after the join, where missing
values will appear, and whether the join keys look consistent across tables.

In [ ]:
# Dtypes and head for the four most important tables
for name in ['orders', 'order_items', 'order_payments', 'customers']:
    df = dfs[name]
    print(f'\n{"-"*55}')
    print(f'  {name}  ({df.shape[0]:,} rows x {df.shape[1]} cols)')
    print(f'{"-"*55}')
    display(df.dtypes.rename('dtype').to_frame().T)
    display(df.head(3))

In [ ]:
# Null rates across all tables — save as a reference table
null_rows = []
for name, df in dfs.items():
    for col, rate in (df.isnull().mean() * 100).items():
        if rate > 0:
            null_rows.append({'table': name, 'column': col, 'null_pct': round(rate, 2)})

null_summary = (
    pd.DataFrame(null_rows)
    .sort_values(['table', 'null_pct'], ascending=[True, False])
    .reset_index(drop=True)
)

null_summary.to_csv(TBL_DIR / '01_null_rates.csv', index=False)
print(f'Saved → {TBL_DIR / "01_null_rates.csv"}')
null_summary

In [ ]:
# Order status breakdown — shows how many orders we will drop in cleaning
status_counts = dfs['orders']['order_status'].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(status_counts.index, status_counts.values,
              color=['#2ecc71' if s == 'delivered' else '#bdc3c7' for s in status_counts.index],
              edgecolor='none')
for bar, val in zip(bars, status_counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 120,
            f'{val:,}', ha='center', va='bottom', fontsize=8)
ax.set_ylabel('Order Count', fontsize=11)
ax.set_title('Order Status Distribution (raw)', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=25)
ax.grid(True, axis='y', linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / '01_order_status_raw.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Clean and Validate

We join the nine tables into one flat item-level master table, cast all date columns
to proper datetime types, then apply three cleaning rules: keep only delivered orders,
drop rows with zero or negative revenue, and drop rows missing a purchase timestamp
or customer ID. After cleaning we engineer cohort month, cohort index, and other
time-based features, then collapse the item-level rows to one row per order.
A final `validate_data` call asserts key invariants — unique order IDs, positive
revenue, and non-negative cohort indices — before anything gets written to disk.

In [ ]:
# Build and cast
master = build_master_table(dfs)
master = cast_dtypes(master)
print(f'Master table (item-level, pre-clean): {master.shape[0]:,} rows x {master.shape[1]} cols')

In [ ]:
# Track rows dropped at each cleaning step
n0 = len(master)
s1 = master[master['order_status'] == 'delivered']
s2 = s1[(s1['payment_value'] > 0) & (s1['price'] > 0)]
s3 = s2.dropna(subset=['order_purchase_timestamp', 'customer_unique_id'])

drop_log = pd.DataFrame({
    'step':         ['raw master', 'keep delivered only', 'keep valid revenue', 'drop null keys'],
    'rows':         [n0, len(s1), len(s2), len(s3)],
    'rows_dropped': [0, n0 - len(s1), len(s1) - len(s2), len(s2) - len(s3)],
})
drop_log['pct_of_raw'] = (drop_log['rows_dropped'] / n0 * 100).round(2)
drop_log.to_csv(TBL_DIR / '01_cleaning_drop_log.csv', index=False)
print(f'Saved → {TBL_DIR / "01_cleaning_drop_log.csv"}\n')
print(drop_log.to_string(index=False))

In [ ]:
# Apply cleaning, feature engineering, and aggregation
master_clean = clean_master_table(master)
master_clean = add_time_features(master_clean)
orders       = add_order_level_features(master_clean)

print(f'After cleaning   (item-level): {master_clean.shape[0]:,} rows')
print(f'After aggregation (order-level): {orders.shape[0]:,} rows x {orders.shape[1]} cols')
orders.head(3)

In [ ]:
# Validate — raises AssertionError if any check fails
orders = validate_data(orders)
print('All validation checks passed.')

In [ ]:
# Revenue and cohort index distributions
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.hist(np.log1p(orders['revenue']), bins=60,
         color='#3498db', edgecolor='none', alpha=0.85)
ax1.set_xlabel('log(1 + Revenue)  [R$]', fontsize=11)
ax1.set_ylabel('Order Count', fontsize=11)
ax1.set_title('Revenue Distribution (log scale)', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.4)
ax1.spines[['top', 'right']].set_visible(False)

cohort_counts = orders['cohort_index'].value_counts().sort_index()
ax2.bar(cohort_counts.index, cohort_counts.values,
        color='#9b59b6', edgecolor='none', width=0.8)
ax2.set_xlabel('Cohort Index (months since first purchase)', fontsize=11)
ax2.set_ylabel('Order Count', fontsize=11)
ax2.set_title('Orders by Cohort Index', fontsize=12, fontweight='bold')
ax2.grid(True, axis='y', linestyle='--', alpha=0.4)
ax2.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / '01_revenue_and_cohort.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Delivery days distribution
delivery = orders['delivery_days'].dropna()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(delivery.clip(upper=delivery.quantile(0.99)), bins=60,
        color='#e67e22', edgecolor='none', alpha=0.85)
ax.axvline(delivery.median(), color='black', linestyle='--', linewidth=1.5,
           label=f'Median: {delivery.median():.0f} days')
ax.axvline(delivery.mean(), color='#c0392b', linestyle=':', linewidth=1.5,
           label=f'Mean: {delivery.mean():.1f} days')
ax.set_xlabel('Delivery Days', fontsize=11)
ax.set_ylabel('Order Count', fontsize=11)
ax.set_title('Delivery Days Distribution (clipped at p99)', fontsize=12, fontweight='bold')
ax.legend(frameon=False, fontsize=9)
ax.grid(True, linestyle='--', alpha=0.4)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / '01_delivery_days.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Save Processed File

The cleaned, validated, order-level table is written to `data/processed/orders_master.parquet`.
Parquet preserves column dtypes (no silent string/float coercions on re-load), reads faster
than CSV, and takes less disk space — all of which matter as the downstream notebooks
load this file repeatedly. We also save a descriptive stats table and a column summary
to `outputs/tables/` for quick reference.

In [ ]:
save_processed(orders, PROC_PATH)

size_kb = PROC_PATH.stat().st_size / 1024
print(f'Saved  →  {PROC_PATH}')
print(f'Size   :  {size_kb:,.1f} KB')
print(f'Rows   :  {orders.shape[0]:,}')
print(f'Cols   :  {orders.shape[1]}')

In [ ]:
# Descriptive stats for numeric columns
numeric_cols = ['revenue', 'item_count', 'avg_item_price',
                'payment_installments', 'delivery_days',
                'review_score', 'cohort_index']

desc = (
    orders[numeric_cols]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    .T.round(2)
)

desc.to_csv(TBL_DIR / '01_descriptive_stats.csv')
print(f'Saved → {TBL_DIR / "01_descriptive_stats.csv"}')
desc

In [ ]:
# Column-level summary of the final processed table
col_summary = pd.DataFrame({
    'dtype':    orders.dtypes.astype(str),
    'non_null': orders.notna().sum(),
    'null_pct': (orders.isna().mean() * 100).round(2),
    'n_unique': orders.nunique(),
})

col_summary.to_csv(TBL_DIR / '01_column_summary.csv')
print(f'Saved → {TBL_DIR / "01_column_summary.csv"}')
col_summary